# Extract Sentiment Features (Sanbase)

This script extracts the following fields:
- social_volume
- social_dominance
- sentiment_positive
- sentiment_negative
- sentiment_polarity
- abnormal_attention

An extract is created per Cryptocurrency coin:
- LTCUSDT
- BTCUSDT
- ETHUSDT
- SOLUSDT
- XRPUSDT
- DOGEUSDT
- BNBUSDT
- ADAUSDT
- LINKUSDT
- AVAXUSDT
- DOTUSDT
- BCHUSDT

In [1]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, UTC
from functools import reduce


# Configuration
API_KEY = "" # insert key here
URL = "https://api.santiment.net/graphql"
headers = {"Authorization": f"Apikey {API_KEY}","Content-Type": "application/json"}
slug_map = {"LTCUSDT": "litecoin",
            "BTCUSDT": "bitcoin",
            "ETHUSDT": "ethereum",
            "SOLUSDT": "solana",
            "XRPUSDT": "ripple",
            "DOGEUSDT": "dogecoin",
            "BNBUSDT": "binance-coin",
            "ADAUSDT": "cardano",
            "LINKUSDT": "chainlink",
            "AVAXUSDT": "avalanche",
            "DOTUSDT": "polkadot",
            "BCHUSDT": "bitcoin-cash"}

# Metrics
metrics = {"social_volume": "social_volume_total",
           "social_dominance": "social_dominance_total",
           "sentiment_positive": "sentiment_positive_total",
           "sentiment_negative": "sentiment_negative_total"}


# Date Ranges
BACKFILL_START_DATE = datetime(2025, 7, 1, tzinfo=UTC)
END_DATE = datetime.now(UTC)
max_retries = 5
base_retry_wait = 10

# Helper Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    parquet_file = f"{symbol}_social_5m.parquet"
    csv_file = f"{symbol}_social_5m.csv"
    checkpoint_file = f"{symbol}_social_checkpoint.parquet"
    return parquet_file, csv_file, checkpoint_file

def load_existing_data(parquet_file):
    if not os.path.exists(parquet_file):
        return pd.DataFrame()

    existing_df = pd.read_parquet(parquet_file)

    if existing_df.empty:
        return pd.DataFrame()

    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True)
    existing_df = (existing_df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))
    return existing_df


def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE

    last_timestamp = existing_df["timestamp"].max()
    incremental_start = last_timestamp
    return incremental_start

def format_santiment_datetime(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def pull_metric(slug,metric_name,output_name,from_date,to_date):

    query = f"""
    {{
      getMetric(metric: "{metric_name}") {{
        timeseriesDataJson(
          slug: "{slug}"
          from: "{from_date}"
          to: "{to_date}"
          interval: "5m"
        )
      }}
    }}
    """

    retries = 0

    while retries < max_retries:
        try:
            response = requests.post(URL,headers=headers,json={"query": query},timeout=60)
            response.raise_for_status()
            data = response.json()

            if "errors" in data:
                print(f"\nERROR: {metric_name}")
                print(data["errors"])
                return None

            rows = (data.get("data", {}).get("getMetric", {}).get("timeseriesDataJson"))

            if not rows:
                print(f"No rows for {metric_name}")
                return None

            df = pd.DataFrame(rows)
            df.rename(columns={"datetime": "timestamp","value": output_name},inplace=True)

            df["timestamp"] = pd.to_datetime(df["timestamp"],utc=True)
            df[output_name] = pd.to_numeric(df[output_name],errors="coerce")
            df = (df[["timestamp", output_name]].drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))

            return df
    return None


def pull_all_metrics_for_period(instrument,slug,from_dt,to_dt):
    from_date = format_santiment_datetime(from_dt)
    to_date = format_santiment_datetime(to_dt)

    print(f"Pulling from: {from_date}")
    print(f"Pulling to:   {to_date}")

    metric_dfs = []

    for output_name, metric_name in metrics.items():
        print(f"\nPulling {metric_name}")

        df = pull_metric(slug=slug,metric_name=metric_name,output_name=output_name,from_date=from_date,to_date=to_date)
        if df is None:
            continue

        metric_dfs.append(df)
        print(f"Rows: {len(df):,}")
        time.sleep(1)

    if len(metric_dfs) == 0:
        print(f"No new social data for {instrument}")
        return pd.DataFrame()

    new_df = reduce(lambda left, right:pd.merge(left,right,on="timestamp",how="outer"),metric_dfs)
    new_df["instrument"] = instrument
    new_df = (new_df.sort_values("timestamp").drop_duplicates(subset=["timestamp"]).reset_index(drop=True))
    return new_df

def calculate_features(final_df):
    if final_df.empty:
        return final_df

    final_df = final_df.copy()
    final_df["timestamp"] = pd.to_datetime(final_df["timestamp"],utc=True)
    final_df = (final_df.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last").reset_index(drop=True))

    required_columns = ["social_volume","social_dominance","sentiment_positive","sentiment_negative"]

    for col in required_columns:
        if col not in final_df.columns:
            final_df[col] = pd.NA

        final_df[col] = pd.to_numeric(final_df[col],errors="coerce")

    # SENTIMENT POLARITY
    final_df["sentiment_polarity"] = (final_df["sentiment_positive"]-final_df["sentiment_negative"])

    # ABNORMAL ATTENTION
    # 288 rows = 24 hours
    window = 288
    rolling_mean = (final_df["social_volume"].rolling(window).mean())
    rolling_std = (final_df["social_volume"].rolling(window).std())
    final_df["abnormal_attention"] = ((final_df["social_volume"] - rolling_mean)/rolling_std)
    final_df["abnormal_attention"] = (final_df["abnormal_attention"].clip(-10, 10))
    return final_df


def extract_social_metrics(instrument,slug):
    parquet_file, csv_file, checkpoint_file = get_output_files(instrument)
    print("\n######################################################################")
    print(f"Starting {instrument}")
    print("#######################################################################")

    existing_df = load_existing_data(parquet_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")

    if not existing_df.empty:
        print(f"Existing latest timestamp: {existing_df['timestamp'].max()}")

    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return

    new_df = pull_all_metrics_for_period(instrument=instrument,slug=slug,from_dt=current_start,to_dt=END_DATE)

    if new_df.empty and existing_df.empty:
        print(f"No final data for {instrument}. Skipping save.")
        return

    if new_df.empty:
        print("\nNo new rows fetched.")
        final_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")

        final_df = pd.concat([existing_df, new_df],ignore_index=True)

    final_df["instrument"] = instrument
    final_df = calculate_features(final_df)
    if final_df.empty:
        print(f"No final data for {instrument}. Skipping save.")
        return
        
    # Save outputs
    final_df.to_parquet(checkpoint_file,index=False)
    final_df.to_parquet(parquet_file,index=False)
    final_df.to_csv(csv_file,index=False)
    print(f"\nFinal shape {instrument}: {final_df.shape}")
    print(f"Final earliest timestamp: {final_df['timestamp'].min()}")
    print(f"Final latest timestamp: {final_df['timestamp'].max()}")
    print(final_df.head())
    print(f"Saved parquet: {parquet_file}")
    print(f"Saved csv: {csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")

# Execute script
for instrument, slug in slug_map.items():
    extract_social_metrics(instrument=instrument,slug=slug)

print("\nAll social extraction complete.")


######################################################################
Starting LTCUSDT
######################################################################
Existing rows: 88,999
Existing latest timestamp: 2026-05-06 00:30:00+00:00
Incremental start: 2026-05-06 00:20:00+00:00
End date: 2026-06-12 20:16:12.488016+00:00
Pulling from: 2026-05-06T00:20:00Z
Pulling to:   2026-06-12T20:16:12Z

Pulling social_volume_total
Rows: 2,256

Pulling social_dominance_total
Rows: 2,256

Pulling sentiment_positive_total
Rows: 2,256

Pulling sentiment_negative_total
Rows: 2,256

New rows before merge: 2,256

Final shape LTCUSDT: (91252, 8)
Final earliest timestamp: 2025-07-01 00:00:00+00:00
Final latest timestamp: 2026-05-13 20:15:00+00:00
   social_volume                 timestamp  social_dominance  \
0            0.0 2025-07-01 00:00:00+00:00          0.000000   
1            1.0 2025-07-01 00:05:00+00:00          1.408451   
2            0.0 2025-07-01 00:10:00+00:00          0.000000   
3        